# Bronze: Ingest NHL Play-by-Play JSON

Loads raw NHL play-by-play payloads into `dbw_hockey_lakehouse.bronze.play_by_play_raw`.

This notebook is meant to run repeatedly as new games become available. It checks which
games already exist in Bronze, fetches only the missing ones from the NHL API, and then
merges the raw payloads into the target table.

A couple design notes:
- this stays fairly raw on purpose since it is Bronze
- the target table stores one JSON payload per game
- merge is used so reruns are safe if the same game gets picked up again


In [ ]:
import json
import requests

from pyspark.sql import functions as F
from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType


In [ ]:
# ---------------- PARAMETERS ----------------
# In practice this would usually be passed in by a job / orchestrator.

dbutils.widgets.text("season", "20242025")

season = dbutils.widgets.get("season").strip()


In [ ]:
# ---------------- ENSURE TARGET TABLE EXISTS ----------------
# Bronze keeps the payload mostly as-is so downstream layers can decide how to parse it.

spark.sql("CREATE SCHEMA IF NOT EXISTS dbw_hockey_lakehouse.bronze")

spark.sql("""
CREATE TABLE IF NOT EXISTS dbw_hockey_lakehouse.bronze.play_by_play_raw (
  game_id BIGINT,
  season STRING,
  source_url STRING,
  raw_json STRING,
  ingested_at TIMESTAMP
) USING delta
""")


In [ ]:
# ---------------- FIND GAMES THAT STILL NEED INGESTION ----------------
# This is the repeatability piece:
# schedule_games tells us what should exist, play_by_play_raw tells us what already exists.
# The left anti join gives us only the missing game_ids for this season.

schedule_games_df = (
    spark.table("dbw_hockey_lakehouse.bronze.schedule_games")
    .where(F.col("season") == season)
    .select(F.col("game_id").cast("bigint").alias("game_id"))
    .distinct()
)

existing_games_df = (
    spark.table("dbw_hockey_lakehouse.bronze.play_by_play_raw")
    .where(F.col("season") == season)
    .select(F.col("game_id").cast("bigint").alias("game_id"))
    .distinct()
)

new_games_df = schedule_games_df.join(existing_games_df, on="game_id", how="left_anti")
game_ids = [int(row["game_id"]) for row in new_games_df.collect()]

print("New games to ingest:", len(game_ids))


In [ ]:
# ---------------- FETCH RAW PLAY-BY-PLAY ----------------
# Source: NHL public API
# One payload per game, stored as raw JSON text in Bronze.

rows = []

for game_id in game_ids:
    source_url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    response = requests.get(source_url, timeout=30)

    if response.status_code != 200:
        # simple skip for now - enough for a Bronze ingestion notebook
        print(f"Skipping game_id={game_id}, status={response.status_code}")
        continue

    payload = response.json()

    rows.append(
        (
            int(game_id),
            season,
            source_url,
            json.dumps(payload),
            None,  # filled just before write
        )
    )

print("Rows collected:", len(rows))


In [ ]:
# ---------------- BUILD STAGING DATAFRAME ----------------
# Keep the schema explicit so the raw landing table stays predictable.

schema = StructType([
    StructField("game_id", LongType(), False),
    StructField("season", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("raw_json", StringType(), False),
    StructField("ingested_at", TimestampType(), True),
])

pbp_raw_df = (
    spark.createDataFrame(rows, schema=schema)
    .drop("ingested_at")
    .withColumn("ingested_at", F.current_timestamp())
)

display(pbp_raw_df.limit(5))


In [ ]:
# ---------------- MERGE INTO BRONZE ----------------
# Merge keeps this rerunnable.
# If a game somehow gets picked up again, we update the raw payload instead of duplicating it.

pbp_raw_df.createOrReplaceTempView("stg_pbp_raw")

spark.sql("""
MERGE INTO dbw_hockey_lakehouse.bronze.play_by_play_raw AS target
USING stg_pbp_raw AS source
ON target.game_id = source.game_id
AND target.season = source.season
WHEN MATCHED THEN UPDATE SET
  target.source_url = source.source_url,
  target.raw_json = source.raw_json,
  target.ingested_at = source.ingested_at
WHEN NOT MATCHED THEN INSERT *
""")


In [ ]:
# ---------------- UPDATE INGESTION STATE ----------------
# Only update the watermark after the merge succeeds.

spark.sql("""
UPDATE dbw_hockey_lakehouse.bronze.ingestion_state
SET last_success_date = current_date(),
    updated_at = current_timestamp()
WHERE pipeline_name = 'nhl_bronze'
""")

print("Watermark updated.")


In [ ]:
# ---------------- QUICK CHECK ----------------

season_count = (
    spark.table("dbw_hockey_lakehouse.bronze.play_by_play_raw")
    .where(F.col("season") == season)
    .count()
)

print("play_by_play_raw count:", season_count)
